In [ ]:
%pip install -q otter-grader

# Case Study 19: Recommendations for a Public Library's E-Media Platform

**DS701 — Session 22 (Wed Nov 18, 2026) — Section A1**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/class_activity_notebooks/19-InClass-Exercise-Recommenders-A1/19-InClass-Exercise-Recommenders-A1.ipynb)

**Time: ~50 minutes.** Work in groups of 2-3.

Parts marked **(autograded)** are submitted to Gradescope; the report-out is graded
for participation. You may use AI assistance, but you must be able to explain and
justify every part of your solution when asked.

**Plan**

| Part | The library's question | Method | Grading | Time |
|---|---|---|---|---|
| 1 | How empty is our borrowing matrix, and how well does "recommend the average" do? | Sparsity; global-mean and title-mean baselines, scored on held-out ratings | autograded | ~5 min |
| 2 | Show us one "patrons who borrowed this also borrowed…" recommendation, computed by hand | Item–item CF: adjusted-cosine similarity, $k$-NN prediction | autograded | ~15 min |
| 3 | Does a model that learns hidden tastes beat the simple ones on ratings we hid — and do its tastes mean anything? | Matrix factorization by ALS; look at the learned factors | autograded | ~20 min |
| — | Report out: what you would tell the library | team memo, then pair-share | participation | ~10 min |

Dependencies: `numpy`, `pandas`, `matplotlib`, `scikit-learn`.


## Case Study Setup

You are a data scientist on contract to a public library system's **e-media
platform** — the app through which patrons borrow e-books, audiobooks and streaming
films. The catalogue team wants two things from you: a *"patrons who borrowed this also
borrowed…"* panel on every title page, and a plan for what a patron who has just signed
up and borrowed nothing should see. What hangs on it: the library pays licence fees per
title, so titles patrons never find are money spent for nothing, and any recommendation
has to be one a librarian can explain to a patron who asks *why am I seeing this?*

Patron borrowing records are confidential and cannot leave the library, so there is no
data to hand you. Instead you build a **simulator** with the shape the catalogue team
described: 80 patrons, 36 titles in three genres, every patron with a favourite genre and
a personal generosity offset (some people rate everything higher), every title with its
own quality offset — the same **biases** the lecture told you to correct for. Ratings are
half-stars from 1 to 5, and about **70% of the matrix is missing** — patrons never rated
those titles. That sparsity is the whole problem. Because *you* wrote the generating
process you know the true answer and can check each method against it; nothing here is
library data, and every number below is a statement about the simulator, not about the
library's patrons.

**What the library asked you, and where you answer it**

1. *How empty is our borrowing matrix, and how well does "recommend the average" do?*
   — Part 1
2. *Show us one "patrons who borrowed this also borrowed…" recommendation computed by
   hand, so a librarian can see where it comes from.* — Part 2
3. *Does a model that learns hidden tastes beat the simple ones on ratings we hid, and do
   the tastes it learns mean anything?* — Part 3
4. *A patron signs up and has borrowed nothing. What do we show her?* — Report out,
   answered from the lecture's cold-start section; no cell in this notebook computes it.

`R_true` is the complete matrix — what every patron *would* rate every title. Like the
species labels in the clustering activities, we never fit on it: it exists only to score
predictions on ratings we hid. `R_train` is what the platform actually has. The simulator,
the 70% of entries that are missing and the 100 ratings held out for scoring are all fixed
by the seed below.

Run the next four cells as-is — everything below depends on them.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

In [ ]:
# Lecture section. Only A1 runs this semester; this is kept because the build
# script reads it to name the variant directory the notebook is published in.
SECTION = "A1"
SEED = 701

In [ ]:
# Colab: fetch the autograder tests for this activity.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    os.makedirs("tests", exist_ok=True)
    BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/class_activity_notebooks/19-InClass-Exercise-Recommenders-A1/tests/"
    for _q in ("q1", "q2", "q3", "q4", "q5"):
        urllib.request.urlretrieve(BASE + _q + ".py", "tests/" + _q + ".py")

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook()

In [ ]:
# --- Build this section's ratings data (planted latent structure; do not edit) ------------
_rng = np.random.default_rng(SEED)
n_users, n_items = 80, 36
GENRES = ["action", "romance", "documentary"]
n_genres = len(GENRES)
genre_of_item = np.repeat(np.arange(n_genres), n_items // n_genres)   # 12 movies per genre

# Genres are directions in a 2-D "taste plane"; each movie is its genre direction + noise;
# each user leans toward one genre. Plus a per-user and a per-item bias.
_angles = np.deg2rad(90.0 + 360.0 * np.arange(n_genres) / n_genres)
_dirs = np.stack([np.cos(_angles), np.sin(_angles)], axis=1)          # (n_genres, 2)
_item_factors = _dirs[genre_of_item] + 0.15 * _rng.normal(size=(n_items, 2))
fav_genre = _rng.integers(0, n_genres, size=n_users)
_user_factors = 1.5 * _dirs[fav_genre] + 0.4 * _rng.normal(size=(n_users, 2))
_user_bias = 0.5 * _rng.normal(size=n_users)
_item_bias = 0.7 * _rng.normal(size=n_items)

def _make_ratings(user_bias, item_bias, user_factors, item_factors, rng):
    r = 3.2 + user_bias[:, None] + item_bias[None, :] + user_factors @ item_factors.T \
        + 0.3 * rng.normal(size=(len(user_bias), len(item_bias)))
    return np.clip(np.round(r * 2) / 2, 1, 5)          # half-star ratings, 1..5

R_true = _make_ratings(_user_bias, _item_bias, _user_factors, _item_factors, _rng)
observed = _rng.random((n_users, n_items)) < 0.30       # ~70% never rated
R = np.where(observed, R_true, np.nan)                  # what the service has collected

# Hide 100 of the observed ratings as a test set: we predict them, then compare with the truth.
_obs_idx = np.argwhere(observed)
holdout = _obs_idx[_rng.choice(len(_obs_idx), 100, replace=False)]   # (100, 2) rows of (user, item)
R_train = R.copy()
R_train[holdout[:, 0], holdout[:, 1]] = np.nan
truth = R_true[holdout[:, 0], holdout[:, 1]]            # the 100 hidden ratings
mask = ~np.isnan(R_train)                               # True where R_train has a rating


def rmse(pred, target):
    pred, target = np.asarray(pred, dtype=float), np.asarray(target, dtype=float)
    return float(np.sqrt(np.mean((pred - target) ** 2)))

print(f"Section {SECTION}: {n_users} users x {n_items} movies, genres = {GENRES}")
print(f"{int(mask.sum())} ratings in R_train, {len(holdout)} held out for scoring")
pd.DataFrame(R_train[:6, :10], columns=[f"m{j}" for j in range(10)]).round(1)

Each row is a patron, each column a title, `NaN` means *no rating*. A quick look at how
the ratings we do have are distributed across patrons and titles.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].imshow(np.where(mask, R_train, 0), aspect="auto", cmap="Blues", vmin=0, vmax=5)
axes[0].set_title("R_train (white = missing)"); axes[0].set_xlabel("movie"); axes[0].set_ylabel("user")
axes[1].bar(np.arange(n_users), np.sort(mask.sum(axis=1))[::-1], color="gray")
axes[1].set_title("ratings per user (sorted)"); axes[1].set_xlabel("user")
axes[2].bar(np.arange(n_items), np.sort(mask.sum(axis=0))[::-1], color="gray")
axes[2].set_title("ratings per movie (sorted)"); axes[2].set_xlabel("movie")
plt.tight_layout(); plt.show()

## Part 1 (autograded): sparsity and two baselines

**Why.** The library's first question is the size of the hole: of all the (patron, title)
pairs, how many have a rating at all? And before any clever method they want to know what
"recommend the average" scores — if a title's average rating already predicts well, the
panel needs nothing cleverer. These two baselines are the numbers every later method has
to beat.

**Task.**

1. `sparsity` — the fraction of the `n_users × n_items` matrix that is **missing** in
   `R_train` (a `float` between 0 and 1).
2. `mu` — the global mean of the observed training ratings.
3. `item_mean` — a length-`n_items` array with each title's mean training rating
   (`np.nanmean` down the columns).
4. `rmse_mu` — RMSE on the 100 held-out ratings if you predict `mu` for every one.
5. `rmse_item_mean` — RMSE if you predict each held-out rating by its title's mean.

Use `holdout[:, 0]` (patrons) and `holdout[:, 1]` (titles) to look things up, and `truth`
for the answers.


In [ ]:
sparsity = ...
mu = ...
item_mean = ...
rmse_mu = ...
rmse_item_mean = ...

print(f"sparsity = {sparsity:.1%}   global mean = {mu:.3f}")
print(f"held-out RMSE: predict mu {rmse_mu:.3f} | predict the movie's mean {rmse_item_mean:.3f}")

In [ ]:
grader.check("q1")

## Part 2 (autograded): item–item collaborative filtering by hand

**Why.** The panel the library wants — *patrons who borrowed this also borrowed…* — is
item–item collaborative filtering, and the catalogue team asked to see one recommendation
computed by hand before they trust a system to produce thousands. The lecture's
short-side rule says this is also the right side to work on: the real platform has far
more patrons than titles, so a title–title similarity table is small, each entry is
estimated from many co-ratings, and it changes slowly enough to precompute.

Item–item CF says: to guess how patron $u$ rates title $i$, find titles **similar to $i$**
that $u$ *has* rated and average those ratings, weighted by similarity. Two ingredients.

**Similarity.** Two titles are similar if the same people rate them the same way. Raw
ratings mislead here — a generous patron pushes every title they touch up together — so
first **subtract each patron's mean** from their ratings (the simplest bias correction),
then compare titles with **cosine similarity** over the mean-centred column vectors,
treating a missing rating as 0 (it contributes nothing to the dot product). This is
"adjusted cosine".

**Task (2a).**

1. `user_mean` — length-`n_users` array of each patron's mean training rating.
2. `R_c` — an `(n_users, n_items)` array: `R_train - user_mean` where a rating exists,
   `0.0` where it is missing (no `NaN`s left).
3. `S` — the `(n_items, n_items)` matrix of cosine similarities between the *columns*
   of `R_c`, i.e. $S_{ij} = \dfrac{c_i^\top c_j}{\lVert c_i\rVert\,\lVert c_j\rVert}$.
   Set the diagonal to `0.0` (a title is not its own neighbour).

Hint: `norms = np.linalg.norm(R_c, axis=0)`; then `R_c.T @ R_c` divided by
`np.outer(norms, norms)`; `np.fill_diagonal`.


In [ ]:
user_mean = ...
R_c = ...

...

print("S.shape =", S.shape, "  similarity range:", S.min().round(2), "to", S.max().round(2))
_i = 0
_top = np.argsort(-S[_i])[:5]
print(f"movie {_i} ({GENRES[genre_of_item[_i]]}) — its 5 most similar movies and their genres:")
print([(int(j), GENRES[genre_of_item[j]], round(float(S[_i, j]), 2)) for j in _top])

In [ ]:
grader.check("q2")

**Prediction.** For a (patron, title) pair, the lecture's bias-corrected item–item
predictor with $k$ nearest neighbours is

$$
\hat r_{ui} \;=\; \bar r_u \;+\; \frac{\sum_{j \in N_k(i,u)} S_{ij}\,(r_{uj} - \bar r_u)}{\sum_{j \in N_k(i,u)} S_{ij}},
$$

where $N_k(i,u)$ is the set of (at most) $k$ titles **rated by $u$** that are most
similar to $i$, keeping only **positive** similarities (the lecture's "common solution
1"). If $u$ has rated no positively-similar title, fall back to $\bar r_u$.

**Task (2b).**

1. Write `predict_cf(u, i, k)` implementing the formula, using `S`, `R_train`, `mask`
   and `user_mean`. (`np.where(mask[u])[0]` gives the titles $u$ rated; the title $i$
   itself is never in that set for a held-out pair, but exclude it anyway to be safe.)
2. `pred_cf` — a length-100 array of predictions for the `holdout` pairs with `K_CF = 10`.
3. `rmse_cf` — its RMSE against `truth`.


In [ ]:
K_CF = 10

def predict_cf(u, i, k=K_CF):
    """Item-item CF prediction of user u's rating of movie i from the k most similar movies u rated."""
    ...


pred_cf = ...
rmse_cf = ...

print(f"held-out RMSE: mu {rmse_mu:.3f} | movie mean {rmse_item_mean:.3f} | item-item CF (k={K_CF}) {rmse_cf:.3f}")
_u, _i = holdout[0]
print(f"user {_u} (likes {GENRES[fav_genre[_u]]}), movie {_i} ({GENRES[genre_of_item[_i]]}): "
      f"truth {truth[0]:.1f}, CF predicts {pred_cf[0]:.2f}, movie mean {item_mean[_i]:.2f}")

**Discuss (participation):** the catalogue team asks how many similar titles the panel
should look at — "2? 5? 20?". Try `k = 2, 5, 20` in the print-out below. Why does the
answer stop mattering so quickly — how many titles has a typical patron in this matrix
rated at all? And when a librarian asks why you subtracted each patron's average *before*
comparing titles rather than after, what do you tell them?

*(1-2 sentences here)*


In [ ]:
for _k in (1, 2, 5, 10, 20):
    print(f"k = {_k:2d}: held-out RMSE {rmse([predict_cf(u, i, _k) for u, i in holdout], truth):.3f}")

In [ ]:
grader.check("q3")

## Part 3 (autograded): matrix factorization with alternating least squares

**Why.** The library's third question is whether a model that *learns* what patrons like
beats the by-hand panel on the ratings we hid — and, since no librarian will act on numbers
they cannot read, whether the "tastes" it learns correspond to anything in the catalogue
(3b). The comparison is honest only on held-out ratings, so `rmse_mf` is the number that
goes in the memo.

CF looks at similarity among titles *or* among patrons. Matrix factorization puts both in
one **latent space**: patron $u$ gets a vector $\mathbf u_u \in \mathbb R^k$, title $i$ gets
$\mathbf v_i \in \mathbb R^k$, and the predicted rating is $\mu + \mathbf u_u^\top \mathbf v_i$.
It is the SVD of Session 14 made to tolerate missing entries: minimise, over the
**observed** entries only,

$$
J(U, V) \;=\; \sum_{(u,i)\ \text{observed}} \big(r_{ui} - \mu - \mathbf u_u^\top \mathbf v_i\big)^2
\;+\; \lambda\big(\lVert U\rVert_F^2 + \lVert V\rVert_F^2\big).
$$

**Alternating least squares.** With $V$ fixed, each patron's row $\mathbf u_u$ is a
separate ridge regression: the design matrix is the rows of $V$ for the titles $u$ rated,
the targets are $u$'s centred ratings,

$$
\mathbf u_u = \big(V_{J_u}^\top V_{J_u} + \lambda I\big)^{-1} V_{J_u}^\top (\mathbf r_u - \mu)_{J_u},
\qquad J_u = \{\text{titles } u \text{ rated}\}.
$$

Then hold $U$ fixed and solve for every title's $\mathbf v_i$ the same way (transpose the
problem). Each half-step can only lower $J$, so alternate until it stops improving.

**Task.**

1. `ridge_rows(Rc, M, F, lam)` — given a centred ratings matrix `Rc` (zeros where missing),
   its boolean mask `M`, the *other* factor matrix `F` (one row per column of `Rc`) and
   `lam`, return the least-squares factor rows: one row per row of `Rc`, computed with the
   formula above using only that row's observed columns (`np.linalg.solve`). A row with
   no observed entries gets zeros.
2. `als(R_train, mask, k, lam, n_iter, tol)` — starting from the given `V0`, alternate
   `U = ridge_rows(Rc, mask, V, lam)` and `V = ridge_rows(Rc.T, mask.T, U, lam)`. After
   each full iteration append the objective $J$ to `obj_hist` and the training RMSE
   (over observed entries, predictions `mu + U @ V.T`) to `rmse_hist`; **stop early**
   when $J$ improves by less than `tol`. Return `U, V, obj_hist, rmse_hist`.
3. Run it with `K_MF = 4`, `LAM = 1.0`, and set `pred_mf` (length-100 predictions for
   the `holdout` pairs) and `rmse_mf`.

Keep the provided `V0` line — the initialisation is fixed so results are reproducible.

In [ ]:
K_MF, LAM = 4, 1.0

def ridge_rows(Rc, M, F, lam):
    """One ridge solve per row of Rc, using only that row's observed columns of F."""
    out = np.zeros((Rc.shape[0], F.shape[1]))
    I = lam * np.eye(F.shape[1])
    for r in range(Rc.shape[0]):
        J = M[r]                          # boolean: which columns of row r are observed
        ...
    return out


def als(R_train, mask, k=K_MF, lam=LAM, n_iter=50, tol=1e-4):
    """Alternating least squares on the observed entries of R_train (global mean removed)."""
    Rc = np.where(mask, R_train - mu, 0.0)
    V = np.random.default_rng(SEED).normal(0.0, 0.1, size=(R_train.shape[1], k))   # V0 -- keep this line
    U = np.zeros((R_train.shape[0], k))
    obj_hist, rmse_hist = [], []
    for it in range(n_iter):
        ...
    return U, V, obj_hist, rmse_hist


U, V, obj_hist, rmse_hist = als(R_train, mask, K_MF, LAM)
pred_mf = ...
rmse_mf = ...

print(f"converged in {len(obj_hist)} iterations; U {U.shape}, V {V.shape}")
print(f"training RMSE {rmse_hist[0]:.3f} -> {rmse_hist[-1]:.3f}")
print(f"held-out RMSE: mu {rmse_mu:.3f} | movie mean {rmse_item_mean:.3f} | item-item CF {rmse_cf:.3f} | MF (k={K_MF}) {rmse_mf:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(obj_hist, "o-", ms=3); ax[0].set_yscale("log"); ax[0].set_title("objective J per ALS iteration"); ax[0].set_xlabel("iteration")
ax[1].plot(rmse_hist, "o-", ms=3, label="training RMSE"); ax[1].axhline(rmse_mf, color="red", ls="--", label="held-out RMSE (final)")
ax[1].set_title("training vs held-out"); ax[1].set_xlabel("iteration"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
grader.check("q4")

### 3b (autograded): do the factors mean anything?

Session 14 promised that low rank *is* latent structure, and the library will want to
hear that the model's factors are tastes, not noise. We planted genres, so we can check:
does the learned title matrix `V` put same-genre titles close together, without ever
having been told the genres?

**Task.**

1. `V_pca` — `V` projected onto its top-2 principal components
   (`PCA(n_components=2).fit_transform(V)`), shape `(n_items, 2)`, for the plot below.
2. `nn_genre_purity` — for each title, find its nearest neighbour **in latent space** by
   cosine similarity of the rows of `V` (not itself), and record whether that neighbour has
   the same genre. Set `nn_genre_purity` to the fraction of titles whose nearest latent
   neighbour shares their genre (a `float`). Chance level is `1 / n_genres`.

Then look at the plot: are the genres separated in a space that was learned from ratings alone?


In [ ]:
V_pca = ...

...

print(f"nearest-latent-neighbour genre purity = {nn_genre_purity:.2f}  (chance = {1 / n_genres:.2f})")

plt.figure(figsize=(6, 5))
for g, name in enumerate(GENRES):
    sel = genre_of_item == g
    plt.scatter(V_pca[sel, 0], V_pca[sel, 1], s=45, label=name)
plt.legend(); plt.grid(alpha=0.3)
plt.title("movies in the learned latent space (top-2 PCs of V)\ncoloured by the planted genre the model never saw")
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.show()

**Discuss (participation):** the catalogue team notices you fitted `K_MF = 4` factors
and asks what four "tastes" there are when the simulator planted only a 2-D taste plane.
What are the other two dimensions doing? (Look at what else went into `_make_ratings`.)
And an engineer on their team asks why you did not simply take the SVD of the ratings
table the way Session 14 did — what do you tell them?

*(1-2 sentences here)*


In [ ]:
grader.check("q5")

<!-- BEGIN QUESTION -->

## Report out: what you would tell the library

Write the two sentences you would put at the top of your memo to the library: the
recommendation, and the one caveat they must hear. Then pair-share with another team.

1. **The recommendation.** Which method should power the "patrons who borrowed this also
   borrowed…" panel, with the held-out RMSE numbers from Parts 1–3 that justify it.
2. **The caveat.** The one thing the catalogue team must hear before they act on your
   number — think about what a held-out RMSE on a simulator does and does not say about
   their patrons.
3. **Their second question, with no code.** A patron signs up and has borrowed nothing.
   From the lecture's cold-start section: what does each of your Part 2 and Part 3
   predictors say about every title for her, and why (point at the formula)? What would
   you put on her home page instead, and which of the lecture's hybrid mitigations would
   you ask the library to build first — onboarding ratings, catalogue metadata, a
   popularity prior, or a blend? Say what each one needs from the library.

**Pair-share (~4 min).** Turn to a neighbouring team and compare memos: did you make the
same recommendation? If not, whose evidence is stronger? Did you name the same caveat?
Be ready to share one takeaway with the class.

> Staff will visit tables during this part. Be ready to explain your reasoning and to
> point at the cell that supports each claim.


_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Wrap-up

Before you leave, each group should be able to answer, without notes:

1. Why is item–item CF's neighbourhood defined by titles *the patron has rated*, and what
   happens to the formula when that set is empty?
2. In ALS, what is being solved when $V$ is held fixed, and why can the objective never
   go up?
3. Why does the lecture's short-side rule point the library at item–item rather than
   user–user — and what is the cheapest thing that beats $\mu$ for a patron with no history?

Submit the autograded parts (1–3) to Gradescope.
